# Sales Demand Forecasting System
Professional AI Case Study Notebook

## Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

## Load Dataset

In [4]:
df = pd.read_csv('Case Study B.csv')
df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10263.0$,34.0$,58.75$,6.0,1997.50,6/28/2004 0:00,Shipped,2.0,6.0,2004.0,...,25593 South Bay Ln.,NaN,Bridgewater,CT,97562,usa,NaN,King,Julie,Small
1,10310.0,49.0,81.4,11.0,3988.60,10/16/2004 0:00,Shipped,4.0,10.0,2004.0,...,Mehrheimerstr. 369,NaN,Koln,NaN,50739,Germany,EMEA,Pfalzheim,Henriette,Medium
2,10109.0,46.0,100.0,5.0,8257.00,3/10/2003 0:00,Shipped,1.0,3.0,2003.0,...,11328 Douglas Av.,NaN,Philadelphia,PA,71270,usa,NaN,Hernandez,Rosa,Large
3,10292.0,35.0,55.07,1.0,1927.45,9/8/2004 0:00,Shipped,3.0,9.0,2004.0,...,897 Long Airport Avenue,NaN,???,NY,10022,USA,NaN,Yu,Kwai,Small
4,10104.0,44.0,39.6,10.0,1742.40,1/31/2003 0:00,Shipped,1.0,1.0,2003.0,...,"C/ Moralzarzal, 86",NaN,???,NaN,28034,Spain,EMEA,Freyre,Diego,Small


## Dataset Info

In [5]:
print(df.shape)
df.info()

(3105, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3105 entries, 0 to 3104
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2832 non-null   object 
 1   QUANTITYORDERED   2832 non-null   object 
 2   PRICEEACH         2832 non-null   object 
 3   ORDERLINENUMBER   2795 non-null   float64
 4   SALES             2795 non-null   float64
 5   ORDERDATE         2795 non-null   object 
 6   STATUS            2795 non-null   object 
 7   QTR_ID            2795 non-null   float64
 8   MONTH_ID          2795 non-null   float64
 9   YEAR_ID           2795 non-null   float64
 10  PRODUCTLINE       2795 non-null   object 
 11  MSRP              2795 non-null   float64
 12  PRODUCTCODE       2795 non-null   object 
 13  CUSTOMERNAME      2795 non-null   object 
 14  PHONE             2795 non-null   object 
 15  ADDRESSLINE1      2795 non-null   object 
 16  ADDRESSLINE2      298 non-null 

## Missing Values

In [6]:
df.isnull().sum()

ORDERNUMBER          273
QUANTITYORDERED      273
PRICEEACH            273
ORDERLINENUMBER      310
SALES                310
ORDERDATE            310
STATUS               310
QTR_ID               310
MONTH_ID             310
YEAR_ID              310
PRODUCTLINE          310
MSRP                 310
PRODUCTCODE          310
CUSTOMERNAME         310
PHONE                310
ADDRESSLINE1         310
ADDRESSLINE2        2807
CITY                 277
STATE               1772
POSTALCODE           383
COUNTRY              210
TERRITORY           1381
CONTACTLASTNAME      310
CONTACTFIRSTNAME     310
DEALSIZE             310
dtype: int64

## Handle Missing Values

In [7]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\sohai\AppData\Local\Temp\ipykernel_12568\2948044075.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


## Remove Duplicates

In [8]:
df.drop_duplicates(inplace=True)

## Date Processing

In [12]:
print(df.columns)

Index(['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER',
       'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID',
       'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE',
       'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',
       'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME',
       'DEALSIZE'],
      dtype='object')


In [15]:
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])

df['Year'] = df['ORDERDATE'].dt.year
df['Month'] = df['ORDERDATE'].dt.month
df['Day'] = df['ORDERDATE'].dt.day
df['Weekday'] = df['ORDERDATE'].dt.weekday

## Feature Engineering

In [17]:
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])

df['Year'] = df['ORDERDATE'].dt.year
df['Month'] = df['ORDERDATE'].dt.month
df['Day'] = df['ORDERDATE'].dt.day
df['Weekday'] = df['ORDERDATE'].dt.weekday

In [19]:
df[['ORDERDATE','Year','Month','Day','Weekday']].head()

,ORDERDATE,Year,Month,Day,Weekday
0,198,1970,1,1,3
1,29,1970,1,1,3
2,120,1970,1,1,3
3,250,1970,1,1,3
4,15,1970,1,1,3


## Encode Categorical Data

In [20]:
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

## Sales Trend Visualization

In [21]:
plt.figure()
plt.plot(df['Date'], df['Sales'])
plt.title("Sales Trend")
plt.show()

KeyError: 'Date'

<Figure size 640x480 with 0 Axes>

## Correlation Heatmap

In [ ]:
plt.figure()
sns.heatmap(df.corr(), cmap='coolwarm')
plt.show()

## Prepare Features

In [ ]:
X = df.drop(['Sales','Date'], axis=1)
y = df['Sales']

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Train Model

In [ ]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

## Prediction

In [ ]:
y_pred = model.predict(X_test)

## Evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

## Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)